# Part B · Token-Level EDA
**Goal:** Analyse how different tokenizers segment Tamil text — expansion ratios, subword fragmentation, unknown-token rates, and feature-engineered signal.

**Data source:** `../part_a_batch_translation/sacrebleu_results.csv` (written by Part A)

In [ ]:
# ── Cell 0 · Runtime check + global visual theme ──────────────────────────────
import subprocess, sys, os, gc, torch, warnings
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import pandas as pd
from IPython.display import display, HTML
warnings.filterwarnings("ignore")

print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("plots", exist_ok=True)

PALETTE = ["#2E86AB", "#A23B72", "#F18F01", "#C73E1D", "#3B1F2B"]
MODEL_COLORS = {
    "IndicTrans2" : "#2E86AB",
    "NLLB-200"    : "#A23B72",
    "mT5"         : "#F18F01",
    "Helsinki"    : "#C73E1D",
    "MADLAD"      : "#3B1F2B",
}
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 150, "figure.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "font.family": "DejaVu Sans",
})
print("\u2713 Global theme applied")

## Why mT5 is included here (but not in BLEU)

mT5 is a **seq2seq pre-training model**, not a translation model — it was not fine-tuned for MT and its raw outputs are not meaningful translations.  
However its **SentencePiece tokenizer** (shared across 101 languages) is architecturally interesting:  
we include mT5 purely to compare tokenisation behaviour (vocabulary coverage, subword fragmentation, expansion ratio) against dedicated MT tokenizers.

In [ ]:
# ── Cell 2 · Load tokenizers (no model weights) ───────────────────────────────
from transformers import AutoTokenizer

TOKENIZER_IDS = {
    "IndicTrans2": ("ai4bharat/indictrans2-en-indic-1B", {"trust_remote_code": True}),
    "NLLB-200":    ("facebook/nllb-200-distilled-600M", {}),
    "mT5":         ("google/mt5-base", {}),
    "Helsinki":    ("Helsinki-NLP/opus-mt-en-ta", {}),
    "MADLAD":      ("google/madlad400-3b-mt", {}),
}

tokenizers = {}
for name, (model_id, kwargs) in TOKENIZER_IDS.items():
    print(f"Loading tokenizer: {name}")
    tokenizers[name] = AutoTokenizer.from_pretrained(model_id, **kwargs)
    print(f"  \u2713 Vocab size: {tokenizers[name].vocab_size:,}")

In [ ]:
# ── Cell 3 · Load Part A translations ─────────────────────────────────────────
df_parta = pd.read_csv("../part_a_batch_translation/sacrebleu_results.csv")
print(f"\u2713 Loaded Part A results: {len(df_parta)} rows")
print(f"  Columns: {list(df_parta.columns)}")

# Handle column name difference between blueprint ("english") and our Part A ("source_english")
eng_col = "english" if "english" in df_parta.columns else "source_english"
sentences_en = df_parta[eng_col].tolist()

model_outputs = {
    "IndicTrans2": df_parta["pred_IndicTrans2"].tolist(),
    "NLLB-200":   df_parta["pred_NLLB-200"].tolist(),
    "Helsinki":   df_parta["pred_Helsinki"].tolist(),
    "MADLAD":     df_parta["pred_MADLAD"].tolist(),
    "mT5":        df_parta["pred_mT5"].tolist(),
}
print("\u2713 All 5 model outputs loaded (including mT5 for tokenization analysis)")

In [ ]:
# ── Cell 4 · Compute token metrics ────────────────────────────────────────────
def clean_tokens(token_list):
    cleaned = []
    for tok in token_list:
        tok = tok.replace("\u2581", "").replace("##", "").strip()
        if tok:
            cleaned.append(tok)
    return cleaned


def compute_token_metrics(text_en, text_ta, tokenizer):
    src_tokens = tokenizer.encode(text_en, add_special_tokens=False)
    tgt_tokens = tokenizer.encode(text_ta, add_special_tokens=False)
    src_count = max(len(src_tokens), 1)
    tgt_count = max(len(tgt_tokens), 1)
    unk_id = tokenizer.unk_token_id
    unk_rate = (tgt_tokens.count(unk_id) / tgt_count * 100) if unk_id else 0.0
    tamil_chars = len(str(text_ta).replace(" ", ""))
    avg_chars_per_tok = tamil_chars / tgt_count
    subword_frag = round(1 / avg_chars_per_tok, 4) if avg_chars_per_tok > 0 else 0
    return {
        "source_token_count": src_count,
        "target_token_count": tgt_count,
        "expansion_ratio": round(tgt_count / src_count, 3),
        "avg_word_length": round(avg_chars_per_tok, 3),
        "subword_fragmentation": subword_frag,
        "unknown_token_rate": round(unk_rate, 3),
    }


records = []
for model_name, translations in model_outputs.items():
    tok = tokenizers[model_name]
    for idx, (en, ta) in enumerate(zip(sentences_en, translations)):
        metrics = compute_token_metrics(en, str(ta), tok)
        records.append({"model": model_name, "sentence_id": idx, "english": en, "tamil": ta, **metrics})

token_df = pd.DataFrame(records)
token_df.to_csv("token_counts.csv", index=False)
print(f"\u2713 Token metrics computed: {len(token_df)} rows")
print(token_df.groupby("model")["expansion_ratio"].mean().round(3))

In [ ]:
# ── Cell 5 · Feature engineering ──────────────────────────────────────────────
token_df["log_expansion"] = np.log1p(token_df["expansion_ratio"])
token_df["efficiency_score"] = token_df["avg_word_length"] / token_df["expansion_ratio"]
token_df["fragmentation_class"] = pd.cut(
    token_df["subword_fragmentation"],
    bins=[0, 0.2, 0.5, 1.0, float("inf")],
    labels=["Low", "Medium", "High", "Very High"],
)
token_df.to_csv("engineered_features.csv", index=False)
print("\u2713 Engineered features saved")

In [ ]:
# ── Cell 6 · VIZ B1 · Radar chart (4 tokenizer metrics) ──────────────────────
METRICS = ["expansion_ratio", "avg_word_length", "subword_fragmentation", "unknown_token_rate"]
LABELS  = ["Expansion\nRatio", "Avg Word\nLength", "Subword\nFragmentation", "Unknown\nToken Rate"]
HIGHER_IS_BETTER = [False, True, False, False]

summary = token_df.groupby("model")[METRICS].mean()
norm = summary.copy()
for col, higher in zip(METRICS, HIGHER_IS_BETTER):
    mn, mx = summary[col].min(), summary[col].max()
    if mx == mn:
        norm[col] = 0.5
    elif higher:
        norm[col] = (summary[col] - mn) / (mx - mn)
    else:
        norm[col] = 1 - (summary[col] - mn) / (mx - mn)

N      = len(METRICS)
angles = [n / N * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
for model_name, color in MODEL_COLORS.items():
    if model_name not in norm.index:
        continue
    values  = norm.loc[model_name].values.flatten().tolist()
    values += values[:1]
    ax.plot(angles, values, "o-", linewidth=2, label=model_name, color=color)
    ax.fill(angles, values, alpha=0.08, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(LABELS, size=10, fontweight="bold")
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["25%", "50%", "75%", "100%"], size=8, color="grey")
ax.grid(color="grey", linestyle="--", linewidth=0.5, alpha=0.5)
ax.set_title(
    "Model Tokenizer Comparison\n(Outer = Better on that metric)",
    size=15, fontweight="bold", pad=25,
)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=11)
plt.tight_layout()
plt.savefig("plots/partb_radar_chart.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# ── Cell 7 · VIZ B2 · Bubble chart (token expansion) ─────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))
for model_name, color in MODEL_COLORS.items():
    sub = token_df[token_df["model"] == model_name]
    ax.scatter(
        sub["source_token_count"], sub["target_token_count"],
        s=sub["expansion_ratio"] * 120, c=color, alpha=0.65,
        edgecolors="white", linewidths=0.8, label=model_name,
    )

xvals = np.linspace(
    token_df["source_token_count"].min(),
    token_df["source_token_count"].max(),
    100,
)
ax.plot(xvals, xvals, "k--", linewidth=1, alpha=0.4, label="Ratio = 1.0")
ax.set_xlabel("Source Token Count (English)", fontsize=13)
ax.set_ylabel("Target Token Count (Tamil)", fontsize=13)
ax.set_title(
    "Token Expansion Bubble Chart\n(Bubble size \u221d expansion ratio)",
    fontsize=15, fontweight="bold",
)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig("plots/partb_bubble_chart.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# ── Cell 8 · VIZ B3 · Violin plot (expansion ratio distribution) ──────────────
fig, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(
    data=token_df, x="model", y="expansion_ratio",
    palette=list(MODEL_COLORS.values()), inner="box", linewidth=1.5, ax=ax,
)
ax.axhline(1.0, color="black", linestyle="--", linewidth=1, alpha=0.5, label="Expansion ratio = 1.0")
ax.set_title("Token Expansion Ratio Distribution by Model", fontsize=15, fontweight="bold")
ax.set_xlabel("Model")
ax.set_ylabel("Expansion Ratio (Target Tokens / Source Tokens)")
ax.legend()
plt.tight_layout()
plt.savefig("plots/partb_violin_expansion.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# ── Cell 9 · VIZ B4 · Heatmap (model × metric summary) ───────────────────────
summary_display = token_df.groupby("model")[METRICS].mean().round(3)

fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(
    summary_display, annot=True, fmt=".2f", cmap="RdYlGn_r",
    linewidths=0.5, linecolor="white", ax=ax,
    cbar_kws={"label": "Metric Value"}, annot_kws={"size": 11, "weight": "bold"},
)
ax.set_title(
    "Model \u00d7 Metric Heatmap (avg across 100 sentences)",
    fontsize=14, fontweight="bold", pad=15,
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.savefig("plots/partb_heatmap.png", bbox_inches="tight", dpi=150)
plt.show()